# পাঠ ১২ - এজেন্ট স্ক্র্যাচপ্যাড সহ চ্যাট ইতিহাস হ্রাস

এই নোটবুকটি দেখায় কিভাবে Microsoft Agent Framework ব্যবহার করে দীর্ঘ কথোপকথনে প্রসঙ্গ (context) পরিচালনা করতে হয়। কথোপকথন বাড়ার সঙ্গে সঙ্গে টোকেনের সংখ্যা বৃদ্ধি পায় — যা শেষ পর্যন্ত মডেলের প্রসঙ্গ উইন্ডোর সীমা ছাড়িয়ে যায়। আমরা এটি সমাধান করি একটি **প্রসঙ্গ সারাংশ প্যাটার্ন** এবং একটি **এজেন্ট স্ক্র্যাচপ্যাড** ব্যবহার করে যা স্থায়ী স্মৃতি হিসেবে কাজ করে।

## আপনি যা শিখবেন:
1. **কেন প্রসঙ্গ পরিচালনা গুরুত্বপূর্ণ**: টোকেন সীমা এবং প্রসঙ্গ উইন্ডোসমূহ বোঝা
2. **প্রসঙ্গ-সচেতন এজেন্ট**: যেগুলো নিজেদের কথোপকথন প্রসঙ্গ পরিচালনা করে এমন এজেন্ট তৈরি করা
3. **প্রসঙ্গ সারাংশ প্যাটার্ন**: কথোপকথন ইতিহাস সংক্ষিপ্ত করার জন্য টুল ব্যবহার
4. **এজেন্ট স্ক্র্যাচপ্যাড**: একটি স্থায়ী স্মৃতি যা প্রসঙ্গ হ্রাসের পরেও থাকে

## প্রয়োজনীয়তা:
- Azure OpenAI সেটআপ এবং পরিবেশ পরিবর্তনশীল কনফিগার করা
- পূর্ববর্তী পাঠ থেকে বেসিক এজেন্ট ধারণা সম্পর্কে দক্ষতা


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## কেন প্রসঙ্গ ব্যবস্থাপনা গুরুত্বপূর্ণ

প্রত্যেক LLM-এর একটি সীমিত **প্রসঙ্গ উইন্ডো** থাকে — একটি একক অনুরোধে এটি সর্বোচ্চ কত সংখ্যক টোকেন প্রক্রিয়া করতে পারে। একটি বহু-ক্রমিক কথোপকথন চলার সঙ্গে:

- প্রতিটি ব্যবহারকারী বার্তা এবং সহকারী প্রত্যুত্তরের সাথে **টোকেনের সংখ্যা সরলরেখায় বৃদ্ধি পায়**।
- **প্রম্পট টোকেন খরচের মূল কারণ** কারণ সম্পূর্ণ ইতিহাস প্রতিটি বারের জন্য পুনরায় পাঠানো হয়।
- অবশেষে কথোপকথন **প্রসঙ্গ উইন্ডো অতিক্রম করে** এবং মডেল হয় সংক্ষিপ্ত করে দেয় বা ত্রুটি দেখায়।

### প্রসঙ্গ ব্যবস্থাপনার কৌশলসমূহ

| কৌশল | এটি কীভাবে কাজ করে | সৌন্দর্যের বিনিময় |
|---|---|---|
| **সংক্ষিপ্তকরণ** | সবচেয়ে পুরানো বার্তাগুলো ফেলে দেয় | প্রাথমিক প্রসঙ্গ হারায় |
| **সারাংশ** | পুরোনো বার্তাগুলো সারাংশে সামারি করে | কিছু বিস্তারিত হারায়, কিন্তু মূল পয়েন্টগুলো বজায় থাকে |
| **স্ক্র্যাচপ্যাড / বাহ্যিক স্মৃতি** | কথোপকথনের বাইরে মূল তথ্য সংরক্ষণ করে | টুল কল প্রয়োজন হয়, কিন্তু যেকোনো হ্রাস সহ্য করে |

এই নোটবুকে আমরা **সারাংশকরণ** এবং একটি **স্ক্র্যাচপ্যাড টুল** একত্রিত করেছি যাতে সংলাপের ইতিহাস সংক্ষিপ্ত করলেও এজেন্ট ধারাবাহিকতা বজায় রাখতে পারে।


## একটি প্রসঙ্গ-সচেতন এজেন্ট তৈরি করা


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## দীর্ঘ কথোপকথনের সিমুলেশন

চলুন একটি বহু-পালার কথোপকথনের মাধ্যমে দেখি কিভাবে প্রসঙ্গ জমা হয়। এজেন্টকে প্রত্যেক পালায় প্রধান বিষয়বস্তু (পছন্দস drop্ভ, বাজেট, ভ্রমণ তারিখ) ধরে রাখতে হবে এবং ধারাবাহিকতা প্রদর্শন করতে হবে।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

মনে করুন এজেন্টটি পূর্ববর্তী কথোপকথন থেকে প্রসঙ্গ ধরে রেখেছে — এটি জানে জাপান সম্পর্কে, সুশি, মন্দির, ফটোগ্রাফি, $৩০০০ বাজেট, একা ভ্রমণ, এবং এপ্রিল সময়সীমা। একটি সংক্ষিপ্ত কথোপকথনে এটি ভালভাবে কাজ করে, তবে কথোপকথন যত বাড়ে পুরো ইতিহাস পুনরায় পাঠানো ব্যয়বহুল হয়ে যায়।

প্রসঙ্গ সঞ্চয়ের জন্য আরও কয়েকটি কথোপকথন নিয়ে চলতে থাকি:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## প্রসঙ্গ সংক্ষিপ্তকরণ প্যাটার্ন

কথোপকথন যত বাড়তে থাকে, আমরা একটি **সংক্ষিপ্তকরণ টুল** ব্যবহার করতে পারি জমাকৃত প্রসঙ্গকে একটি সুসংক্ষিপ্ত ফরম্যাটে রূপান্তর করার জন্য। এজেন্ট এই টুলটি কল করে মূল পছন্দসমূহ রেকর্ড করে যাতে পুরোনো মেসেজগুলি বাদ পড়লেও গুরুত্বপূর্ণ তথ্য সংরক্ষিত থাকে।

এই প্যাটার্নটি আরো জটিল ইতিহাস হ্রাসের জন্য ভিত্তি গঠন করে:
1. এজেন্ট কথোপকথন থেকে মূল তথ্য সনাক্ত করে
2. এটি সংক্ষিপ্তকরণ টুল কল করে সেগুলি সংরক্ষণ করে
3. পুরোনো মেসেজগুলি নিরাপদে সরানো যায় কারণ সংক্ষিপ্তকরণ যা গুরুত্বপূর্ণ তা ধারণ করে

নীচে আমরা একটি `summarize_preferences` টুল সংজ্ঞায়িত করেছি যা এজেন্ট কল করতে পারে যা সে শিখেছে তার একটি সংক্ষিপ্ত সারাংশ রেকর্ড করার জন্য।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## সারাংশ

এই পাঠে আপনি শেখেন কিভাবে Microsoft Agent Framework ব্যবহার করে দীর্ঘগামী এজেন্ট কথোপকথনে প্রাসঙ্গিকতা পরিচালনা করবেন:

### প্রধান ধারণাসমূহ
- **প্রাসঙ্গিকতা জানালা সীমিত** — কথোপকথনের ইতিহাসের প্রতিটি টোকেনের প্রকৃত খরচ হয় এবং এটি সীমার মধ্যে গণ্য হয়।
- **সারাংশ তৈরির সরঞ্জাম** এজেন্টকে জমাকৃত প্রাসঙ্গিকতাকে সংক্ষিপ্ত সারাংশে রূপান্তর করতে দেয়, যা টোকেন ব্যবহারে কমিয়ে essential তথ্য সংরক্ষণ করে।
- **এজেন্ট স্ক্র্যাচপ্যাড** দেওয়া হয় একটি স্থায়ী বাহ্যিক মেমরি যা যেকোনো কথোপকথনের কমানোর পরেও টিকে থাকে।

### আপনি যা তৈরি করেছেন
- একটি **প্রাসঙ্গিক-সচেতন এজেন্ট** যা বহু-বার্তালাপের মধ্যে ধারাবাহিকতা বজায় রাখে
- একটি **সারাংশ সরঞ্জাম** (`summarize_preferences`) যা ব্যবহারকারীর মূল তথ্য সংক্ষেপে রেকর্ড করে
- একটি **বহু-বার্তালাপ** যা প্রাসঙ্গিকতা রক্ষার ও পরিবর্তন পরিচালনার উদাহরণ দেয়

### বাস্তব জীবন ব্যবহারের ক্ষেত্রসমূহ
- **গ্রাহক সেবা বট**: দীর্ঘ সাপোর্ট সেশনের মধ্যে পছন্দ স্মরণ রাখে
- **ব্যক্তিগত সহকারী**: প্রক্রিয়াধীন প্রকল্প অনুসরণ করে নতুন করে প্রাসঙ্গিকতা ব্যাখ্যা না করে
- **শিক্ষাগত শিক্ষক**: অনেক কথোপকথন জুড়ে ছাত্রের অগ্রগতি বজায় রাখে

### পরবর্তী ধাপ
- ফাইল-ভিত্তিক স্থায়িত্বসহ পূর্ণ স্ক্র্যাচপ্যাড সরঞ্জাম বাস্তবায়ন করা
- সারাংশ তৈরির পর স্বয়ংক্রিয়ভাবে ইতিহাস সংক্ষিপ্তকরণ যোগ করা
- সেমান্টিক মেমরি অনুসন্ধানের জন্য ভেক্টর ডাটাবেসের সাথে সংযোগ করা
- পূর্ণ প্রাসঙ্গিকতা সহ দিন পরদিন কথোপকথন পুনরায় শুরু করার জন্য এজেন্ট তৈরি করা


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকারোক্তি**:  
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসম্ভব সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসঙ্গতি থাকতে পারে। মৌলিক নথির নিজস্ব ভাষার সংস্করণই কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করতে হবে। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানুষ দ্বারা অনুবাদ করানোই সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে কোনও ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা কোনো দায়বদ্ধ থাকব না।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
